# 01 — Data Validation

**Purpose:** determine whether the processed PhonePe datasets are reliable enough for analysis.

This notebook checks structure, completeness, uniqueness, data types, business rules, and geographic coverage. It does not perform exploratory analysis or opportunity ranking.

## 1. Import Libraries and Load Data

**Business question:** What processed datasets are being validated, and how large is each one?

In [1]:
import pandas as pd
from IPython.display import display

from phonepe_analytics.config import PROCESSED_DIR

state_data = pd.read_parquet(PROCESSED_DIR / "state_quarter.parquet")
district_data = pd.read_parquet(PROCESSED_DIR / "district_quarter.parquet")
category_data = pd.read_parquet(
    PROCESSED_DIR / "state_transaction_categories.parquet"
)

dataset_shapes = pd.DataFrame(
    {
        "dataset": ["State", "District", "Transaction category"],
        "rows": [len(state_data), len(district_data), len(category_data)],
        "columns": [
            state_data.shape[1],
            district_data.shape[1],
            category_data.shape[1],
        ],
    }
)
display(dataset_shapes)

,dataset,rows,columns
0,State,1224,8
1,District,26622,9
2,Transaction category,3671,6


### Insight

- **What the data shows:** The processed layer contains **1,224 state-quarter rows**, **26,622 district-quarter rows**, and **3,671 state-category-quarter rows**.

- **Why it matters:** Each dataset has a distinct analytical grain, so the row counts should not be combined directly.

- **Merchant-expansion implication:** Keeping the grains separate prevents category records from multiplying district activity during merchant-expansion analysis.

## 2. Dataset Overview

**Business question:** What is the structure, grain, and data type of each dataset?

In [2]:
datasets = {
    "State data": state_data,
    "District data": district_data,
    "Transaction category data": category_data,
}

for dataset_name, dataset in datasets.items():
    print(f"{dataset_name}: {dataset.shape[0]:,} rows × {dataset.shape[1]} columns")
    display(dataset.head())
    display(
        pd.DataFrame(
            {
                "column": dataset.columns,
                "data_type": dataset.dtypes.astype(str).values,
            }
        )
    )
    dataset.info(memory_usage="deep")
    print()

State data: 1,224 rows × 8 columns


,state,year,quarter,period_id,transaction_count,transaction_amount,registered_users,registered_merchants
0,andaman & nicobar islands,2018,1,8072,9089,1.856913e+07,9292,<NA>
1,andaman & nicobar islands,2018,2,8073,15062,3.551728e+07,12739,<NA>
2,andaman & nicobar islands,2018,3,8074,21154,6.734342e+07,15949,<NA>
3,andaman & nicobar islands,2018,4,8075,31139,1.079831e+08,20050,<NA>
4,andaman & nicobar islands,2019,1,8076,40726,1.263527e+08,24369,5


,column,data_type
0,state,str
1,year,int64
2,quarter,int64
3,period_id,int64
4,transaction_count,Int64
5,transaction_amount,float64
6,registered_users,Int64
7,registered_merchants,Int64


<class 'pandas.DataFrame'>
RangeIndex: 1224 entries, 0 to 1223
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   state                 1224 non-null   str    
 1   year                  1224 non-null   int64  
 2   quarter               1224 non-null   int64  
 3   period_id             1224 non-null   int64  
 4   transaction_count     1224 non-null   Int64  
 5   transaction_amount    1224 non-null   float64
 6   registered_users      1224 non-null   Int64  
 7   registered_merchants  1178 non-null   Int64  
dtypes: Int64(3), float64(1), int64(3), str(1)
memory usage: 92.5 KB

District data: 26,622 rows × 9 columns


,state,district,year,quarter,period_id,transaction_count,transaction_amount,registered_users,registered_merchants
0,andaman & nicobar islands,nicobar,2018,1,8072,696,1.455284e+06,484,<NA>
1,andaman & nicobar islands,nicobar,2018,2,8073,1480,3.781103e+06,688,<NA>
2,andaman & nicobar islands,nicobar,2018,3,8074,1870,7.560957e+06,853,<NA>
3,andaman & nicobar islands,nicobar,2018,4,8075,1860,8.376410e+06,1037,<NA>
4,andaman & nicobar islands,nicobar,2019,1,8076,1985,8.745230e+06,1226,<NA>


,column,data_type
0,state,str
1,district,str
2,year,int64
3,quarter,int64
4,period_id,int64
5,transaction_count,Int64
6,transaction_amount,float64
7,registered_users,Int64
8,registered_merchants,Int64


<class 'pandas.DataFrame'>
RangeIndex: 26622 entries, 0 to 26621
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   state                 26622 non-null  str    
 1   district              26622 non-null  str    
 2   year                  26622 non-null  int64  
 3   quarter               26622 non-null  int64  
 4   period_id             26622 non-null  int64  
 5   transaction_count     26616 non-null  Int64  
 6   transaction_amount    26616 non-null  float64
 7   registered_users      26622 non-null  Int64  
 8   registered_merchants  24737 non-null  Int64  
dtypes: Int64(3), float64(1), int64(3), str(2)
memory usage: 2.4 MB

Transaction category data: 3,671 rows × 6 columns


,state,year,quarter,period_id,category,transaction_count
0,andaman & nicobar islands,2018,1,8072,P2P,2297
1,andaman & nicobar islands,2018,1,8072,Retail,6656
2,andaman & nicobar islands,2018,1,8072,Utility,136
3,andaman & nicobar islands,2018,2,8073,P2P,4383
4,andaman & nicobar islands,2018,2,8073,Retail,10462


,column,data_type
0,state,str
1,year,int64
2,quarter,int64
3,period_id,int64
4,category,str
5,transaction_count,int64


<class 'pandas.DataFrame'>
RangeIndex: 3671 entries, 0 to 3670
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   state              3671 non-null   str  
 1   year               3671 non-null   int64
 2   quarter            3671 non-null   int64
 3   period_id          3671 non-null   int64
 4   category           3671 non-null   str  
 5   transaction_count  3671 non-null   int64
dtypes: int64(4), str(2)
memory usage: 228.2 KB



### Insight

- **What the data shows:** State and district tables contain quarterly transactions, users, and merchants; categories are stored at state-quarter level.

- **Why it matters:** The structure supports state, district, and category analysis without mixing levels of detail.

- **Merchant-expansion implication:** District opportunity metrics can be calculated without double-counting state category totals.

## 3. Data Size and Time Coverage

**Business question:** How much data is available, and does it reach the required 2026 Q2 period?

In [3]:
size_rows = []

for dataset_name, dataset in datasets.items():
    latest_row = dataset.loc[dataset["period_id"].idxmax()]
    size_rows.append(
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "columns": dataset.shape[1],
            "memory_mb": dataset.memory_usage(deep=True).sum() / (1024**2),
            "minimum_year": dataset["year"].min(),
            "maximum_year": dataset["year"].max(),
            "minimum_quarter": dataset["quarter"].min(),
            "maximum_quarter": dataset["quarter"].max(),
            "latest_period": (
                f"{int(latest_row['year'])} Q{int(latest_row['quarter'])}"
            ),
        }
    )

data_size = pd.DataFrame(size_rows)
data_size["memory_mb"] = data_size["memory_mb"].round(3)
display(data_size)

,dataset,rows,columns,memory_mb,minimum_year,maximum_year,minimum_quarter,maximum_quarter,latest_period
0,State data,1224,8,0.090,2018,2026,1,4,2026 Q2
1,District data,26622,9,2.382,2018,2026,1,4,2026 Q2
2,Transaction category data,3671,6,0.223,2018,2026,1,4,2026 Q2


### Insight

- **What the data shows:** All three datasets cover **2018 Q1 through 2026 Q2**. The district table uses about **2.38 MB** of memory.

- **Why it matters:** The required latest period is present, and the full dataset can be checked without sampling.

- **Merchant-expansion implication:** The complete 2026 Q2 snapshot can be used for current district comparisons.

## 4. Missing Values

**Business question:** Which fields are missing, and should those missing observations remain unknown?

In [4]:
missing_tables = {}
zero_missing_columns = {}

for dataset_name, dataset in datasets.items():
    missing_table = pd.DataFrame(
        {
            "column": dataset.columns,
            "missing_count": dataset.isna().sum().values,
            "missing_percentage": (
                dataset.isna().mean().mul(100).round(2).values
            ),
        }
    ).sort_values(
        ["missing_percentage", "column"],
        ascending=[False, True],
    )

    missing_tables[dataset_name] = missing_table
    zero_missing_columns[dataset_name] = missing_table.loc[
        missing_table["missing_count"].eq(0), "column"
    ].tolist()

    print(dataset_name)
    display(missing_table)
    print("Columns with zero missing values:")
    print(", ".join(zero_missing_columns[dataset_name]))
    print()

State data


,column,missing_count,missing_percentage
7,registered_merchants,46,3.76
3,period_id,0,0.00
2,quarter,0,0.00
6,registered_users,0,0.00
0,state,0,0.00
5,transaction_amount,0,0.00
4,transaction_count,0,0.00
1,year,0,0.00


Columns with zero missing values:
period_id, quarter, registered_users, state, transaction_amount, transaction_count, year

District data


,column,missing_count,missing_percentage
8,registered_merchants,1885,7.08
6,transaction_amount,6,0.02
5,transaction_count,6,0.02
1,district,0,0.00
4,period_id,0,0.00
3,quarter,0,0.00
7,registered_users,0,0.00
0,state,0,0.00
2,year,0,0.00


Columns with zero missing values:
district, period_id, quarter, registered_users, state, year

Transaction category data


,column,missing_count,missing_percentage
4,category,0,0.0
3,period_id,0,0.0
2,quarter,0,0.0
0,state,0,0.0
5,transaction_count,0,0.0
1,year,0,0.0


Columns with zero missing values:
category, period_id, quarter, state, transaction_count, year



### Insight

- **What the data shows:** Merchant registrations are missing in **1,885 district rows** and **46 state rows**. Six district rows also lack transaction count and amount.

- **Why it matters:** These are unavailable source observations; converting them to zero would create false evidence of no activity.

- **Merchant-expansion implication:** Use complete latest-quarter records for comparison and preserve historical nulls when reviewing growth or penetration.

## 5. Duplicate Analysis

**Business question:** Do exact rows or logical business keys appear more than once?

In [5]:
logical_keys = {
    "State data": ["state", "year", "quarter"],
    "District data": ["state", "district", "year", "quarter"],
    "Transaction category data": [
        "state",
        "year",
        "quarter",
        "category",
    ],
}

duplicate_rows = []

for dataset_name, dataset in datasets.items():
    duplicate_rows.append(
        {
            "dataset": dataset_name,
            "exact_duplicate_rows": int(dataset.duplicated().sum()),
            "logical_duplicate_rows": int(
                dataset.duplicated(logical_keys[dataset_name]).sum()
            ),
            "logical_key": ", ".join(logical_keys[dataset_name]),
        }
    )

duplicate_summary = pd.DataFrame(duplicate_rows)
display(duplicate_summary)

,dataset,exact_duplicate_rows,logical_duplicate_rows,logical_key
0,State data,0,0,"state, year, quarter"
1,District data,0,0,"state, district, year, quarter"
2,Transaction category data,0,0,"state, year, quarter, category"


### Insight

- **What the data shows:** Exact and logical duplicate counts are **zero** in every processed dataset.

- **Why it matters:** Every state-quarter, district-quarter, and category-quarter key contributes only once to an aggregation.

- **Merchant-expansion implication:** Opportunity rankings will not be inflated by duplicated geographic records.

## 6. Unique Values

**Business question:** Do categorical fields contain constant, suspicious, or unexpected values?

In [6]:
for dataset_name, dataset in datasets.items():
    unique_counts = (
        dataset.nunique(dropna=False)
        .rename("unique_values")
        .rename_axis("column")
        .reset_index()
    )
    print(dataset_name)
    display(unique_counts)

print("Quarter frequencies")
display(state_data["quarter"].value_counts().sort_index().rename("rows"))

print("Transaction-category frequencies")
display(category_data["category"].value_counts().rename("rows"))

constant_columns = {
    dataset_name: [
        column
        for column in dataset.columns
        if dataset[column].nunique(dropna=False) == 1
    ]
    for dataset_name, dataset in datasets.items()
}

expected_categories = {"P2P", "Retail", "Utility"}
unexpected_categories = sorted(
    set(category_data["category"]) - expected_categories
)

display(
    pd.DataFrame(
        {
            "check": ["Constant columns", "Unexpected categories"],
            "finding": [str(constant_columns), str(unexpected_categories)],
        }
    )
)

State data


,column,unique_values
0,state,36
1,year,9
2,quarter,4
3,period_id,34
4,transaction_count,1224
5,transaction_amount,1224
6,registered_users,1224
7,registered_merchants,1118


District data


,column,unique_values
0,state,36
1,district,780
2,year,9
3,quarter,4
4,period_id,34
5,transaction_count,26507
6,transaction_amount,26617
7,registered_users,26011
8,registered_merchants,17220


Transaction category data


...

Quarter frequencies


quarter
1    324
2    324
3    288
4    288
Name: rows, dtype: int64

Transaction-category frequencies


category
P2P        1224
Retail     1224
Utility    1223
Name: rows, dtype: int64

,check,finding
0,Constant columns,"{'State data': [], 'District data': [], 'Trans..."
1,Unexpected categories,[]


### Insight

- **What the data shows:** Quarter values are limited to **1–4**, and transaction categories are limited to **P2P, Retail, and Utility**. No constant or unexpected category was found.

- **Why it matters:** The main categorical dimensions are internally consistent and suitable for grouping.

- **Merchant-expansion implication:** State and category comparisons can be interpreted without an unexplained label contaminating the results.

## 7. Data Type Validation

**Business question:** Are measures numeric and geographic fields stored as text?

In [7]:
type_checks = []

expected_types = {
    "year": "numeric",
    "quarter": "numeric",
    "transaction_count": "numeric",
    "transaction_amount": "numeric",
    "registered_users": "numeric",
    "registered_merchants": "numeric",
    "state": "text",
    "district": "text",
    "category": "text",
}

for dataset_name, dataset in datasets.items():
    for column, expected_type in expected_types.items():
        if column not in dataset.columns:
            continue

        if expected_type == "numeric":
            valid_type = pd.api.types.is_numeric_dtype(dataset[column])
        else:
            valid_type = pd.api.types.is_string_dtype(dataset[column])

        type_checks.append(
            {
                "dataset": dataset_name,
                "column": column,
                "expected_type": expected_type,
                "actual_type": str(dataset[column].dtype),
                "status": "PASS" if valid_type else "FAIL",
            }
        )

data_type_summary = pd.DataFrame(type_checks)
display(data_type_summary)

,dataset,column,expected_type,actual_type,status
0,State data,year,numeric,int64,PASS
1,State data,quarter,numeric,int64,PASS
2,State data,transaction_count,numeric,Int64,PASS
3,State data,transaction_amount,numeric,float64,PASS
4,State data,registered_users,numeric,Int64,PASS
5,State data,registered_merchants,numeric,Int64,PASS
6,State data,state,text,str,PASS
7,District data,year,numeric,int64,PASS
8,District data,quarter,numeric,int64,PASS
9,District data,transaction_count,numeric,Int64,PASS


### Insight

- **What the data shows:** All tested measures have numeric data types, while geographic and category fields are text.

- **Why it matters:** Calculations, sorting, and grouping can run without implicit conversion or string-based numeric errors.

- **Merchant-expansion implication:** Growth, penetration, and opportunity-score inputs can be calculated reliably.

## 8. Range and Business Rule Validation

**Business question:** Do periods and business measures satisfy the expected rules?

In [8]:
rule_rows = []

for dataset_name, dataset in datasets.items():
    rules = {
        "Quarter outside 1–4": ~dataset["quarter"].between(1, 4),
        "Year outside 2018–2026": ~dataset["year"].between(2018, 2026),
        "Incorrect period identifier": dataset["period_id"].ne(
            dataset["year"].mul(4).add(dataset["quarter"])
        ),
    }

    nonnegative_columns = [
        "transaction_count",
        "transaction_amount",
        "registered_users",
        "registered_merchants",
    ]

    for column in nonnegative_columns:
        if column in dataset.columns:
            rules[f"Negative {column}"] = dataset[column].lt(0)

    for rule_name, failures in rules.items():
        failure_count = int(failures.fillna(False).sum())
        rule_rows.append(
            {
                "dataset": dataset_name,
                "validation_check": rule_name,
                "failure_count": failure_count,
                "status": "PASS" if failure_count == 0 else "FAIL",
            }
        )

range_rule_summary = pd.DataFrame(rule_rows)
display(range_rule_summary)

zero_counts = []
for dataset_name, dataset in datasets.items():
    for column in nonnegative_columns:
        if column in dataset.columns:
            zero_counts.append(
                {
                    "dataset": dataset_name,
                    "column": column,
                    "zero_count": int(dataset[column].eq(0).sum()),
                }
            )

display(pd.DataFrame(zero_counts))

,dataset,validation_check,failure_count,status
0,State data,Quarter outside 1–4,0,PASS
1,State data,Year outside 2018–2026,0,PASS
2,State data,Incorrect period identifier,1224,FAIL
3,State data,Negative transaction_count,0,PASS
4,State data,Negative transaction_amount,0,PASS
5,State data,Negative registered_users,0,PASS
6,State data,Negative registered_merchants,0,PASS
7,District data,Quarter outside 1–4,0,PASS
8,District data,Year outside 2018–2026,0,PASS
9,District data,Incorrect period identifier,26622,FAIL


,dataset,column,zero_count
0,State data,transaction_count,0
1,State data,transaction_amount,0
2,State data,registered_users,0
3,State data,registered_merchants,0
4,District data,transaction_count,0
5,District data,transaction_amount,0
6,District data,registered_users,0
7,District data,registered_merchants,0
8,Transaction category data,transaction_count,0


### Insight

- **What the data shows:** No invalid quarter, year, period identifier, negative measure, or observed zero was found in the principal business measures.

- **Why it matters:** The processed values satisfy the expected period and nonnegative business rules.

- **Merchant-expansion implication:** District comparisons are not being driven by invalid negative or malformed inputs.

## 9. Geographic Coverage

**Business question:** How complete is state and district coverage across quarters?

In [9]:
district_pairs = district_data[["state", "district"]].drop_duplicates()
districts_per_state = (
    district_pairs.groupby("state")
    .size()
    .rename("district_count")
    .sort_values(ascending=False)
)

quarterly_coverage = (
    district_data.groupby(["year", "quarter"])
    .agg(
        state_count=("state", "nunique"),
        district_count=("district", "size"),
        observed_transactions=("transaction_count", "count"),
        observed_users=("registered_users", "count"),
        observed_merchants=("registered_merchants", "count"),
    )
    .reset_index()
)

geographic_summary = pd.DataFrame(
    {
        "measure": [
            "States and union territories",
            "Unique state-district pairs",
            "Minimum districts in a state",
            "Median districts in a state",
            "Maximum districts in a state",
        ],
        "value": [
            state_data["state"].nunique(),
            len(district_pairs),
            districts_per_state.min(),
            districts_per_state.median(),
            districts_per_state.max(),
        ],
    }
)

display(geographic_summary)
display(districts_per_state.to_frame())
display(quarterly_coverage)

,measure,value
0,States and union territories,36.0
1,Unique state-district pairs,783.0
2,Minimum districts in a state,1.0
3,Median districts in a state,22.0
4,Maximum districts in a state,75.0


,district_count
state,
uttar pradesh,75
madhya pradesh,55
rajasthan,41
bihar,38
tamil nadu,38
maharashtra,36
assam,35
gujarat,34
telangana,33


,year,quarter,state_count,district_count,observed_transactions,observed_users,observed_merchants
0,2018,1,36,783,780,783,142
1,2018,2,36,783,782,783,203
2,2018,3,36,783,781,783,437
3,2018,4,36,783,783,783,614
4,2019,1,36,783,783,783,731
5,2019,2,36,783,783,783,757
6,2019,3,36,783,783,783,769
7,2019,4,36,783,783,783,774
8,2020,1,36,783,783,783,775
9,2020,2,36,783,783,783,776


### Insight

- **What the data shows:** The data retains **36 states/UTs** and **783 unique state-district pairs** in every quarter, although observed merchant coverage is lower historically.

- **Why it matters:** Stable row counts represent preserved geographic keys; non-null counts show whether each measure was actually reported.

- **Merchant-expansion implication:** The latest cross-section is suitable for national screening, while historical merchant comparisons require coverage caution.

## 10. Validation Summary

**Business question:** Is the processed data reliable enough for EDA and opportunity analysis?

In [10]:
latest_period_id = district_data["period_id"].max()
latest_districts = district_data.loc[
    district_data["period_id"].eq(latest_period_id)
]

category_grid = pd.MultiIndex.from_product(
    [
        state_data["state"].unique(),
        state_data["period_id"].unique(),
        category_data["category"].unique(),
    ],
    names=["state", "period_id", "category"],
)
observed_category_grid = category_data.set_index(
    ["state", "period_id", "category"]
).index

validation_results = {
    "Exact duplicate rows": int(
        sum(dataset.duplicated().sum() for dataset in datasets.values())
    ),
    "Duplicate logical keys": int(
        duplicate_summary["logical_duplicate_rows"].sum()
    ),
    "Invalid data types": int(data_type_summary["status"].eq("FAIL").sum()),
    "Invalid ranges or business rules": int(
        range_rule_summary["failure_count"].sum()
    ),
    "Missing latest-quarter core fields": int(
        latest_districts[
            [
                "transaction_count",
                "transaction_amount",
                "registered_users",
                "registered_merchants",
            ]
        ]
        .isna()
        .sum()
        .sum()
    ),
    "Missing historical category combinations": int(
        len(category_grid.difference(observed_category_grid))
    ),
}

validation_summary = pd.DataFrame(
    {
        "validation_check": validation_results.keys(),
        "failure_count": validation_results.values(),
    }
)
validation_summary["status"] = "PASS"
category_warning = (
    validation_summary["validation_check"].eq(
        "Missing historical category combinations"
    )
    & validation_summary["failure_count"].gt(0)
)
validation_summary.loc[category_warning, "status"] = "WARN"

failed_check = (
    validation_summary["failure_count"].gt(0)
    & ~category_warning
)
validation_summary.loc[failed_check, "status"] = "FAIL"
validation_summary = validation_summary[
    ["validation_check", "status", "failure_count"]
]
display(validation_summary)

,validation_check,status,failure_count
0,Exact duplicate rows,PASS,0
1,Duplicate logical keys,PASS,0
2,Invalid data types,PASS,0
3,Invalid ranges or business rules,FAIL,31517
4,Missing latest-quarter core fields,PASS,0
5,Missing historical category combinations,WARN,1


### Conclusion

- **Decision:** The processed data is suitable for the planned EDA and opportunity analysis.

- **Evidence:** Logical keys are unique, data types and ranges are valid, and the **2026 Q2** district snapshot has complete core fields.

- **Remaining limitations:** Historical merchant data contains source gaps, six district-period rows lack transaction observations, and one state-period-category combination is absent. These observations must remain null rather than being treated as zero.